# 1.환경준비

In [ ]:
# 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format = 'retina'

In [ ]:
df = pd.read_csv("/content/titanic_train.csv")
df.head()

# 2.데이터 이해

In [ ]:
# 상위 몇 개 행 확인
df.head()

In [ ]:
# 변수 확인
df.info()

In [ ]:
# 기초 통계
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.corr(numeric_only=True)

# 3. 데이터 전처리

**1) 변수 제거**

In [ ]:
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']

df.drop(drop_cols, axis=1, inplace=True)

**2) 결측치 처리**

In [ ]:
# Age 결측치 중앙값으로 채움
age_median = df['Age'].median()
df['Age'].fillna(age_median, inplace=True)

In [ ]:
# Embarked 최빈값으로 채우기
emb_freq = df['Embarked'].mode()[0]
df['Embarked'].fillna(emb_freq, inplace=True)

**3) x, y 분리**

In [ ]:
target = 'Survived'

X = df.drop(target, axis=1)
y = df[target]

**4) 가변수화**

In [ ]:
dumm_cols = ['Pclass', 'Sex', 'Embarked']

X = pd.get_dummies(X, columns=dumm_cols, drop_first=True)

X.head()

**5) 학습용, 평가용 데이터 분리**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=11)

# 4.모델링

In [ ]:
# 1단계: 불러오기
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# 2단계: 선언하기
model = DecisionTreeClassifier(max_depth=5)

In [ ]:
# 3단계: 학습하기
model.fit(X_train, y_train)

In [ ]:
# 4단계: 예측하기
y_pred = model.predict(X_test)

## 4.1.Confusion Matrix : 오차행렬

In [ ]:
# 5단계 평가하기
# 예측결과와 실제 결과의 오차행렬 출력
print(confusion_matrix(y_test, y_pred))

## 4.2.정밀도(Prescision) 재현율(Recall) 구해보기

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, classification_report

print("정밀도 : ", precision_score(y_test, y_pred))
print("재현율 : ", recall_score(y_test, y_pred))

In [ ]:
# 함수로 만들어보기

def metrics_eval(y_test, pred):
    print("정확도 : ", accuracy_score(y_test, pred))
    print("정밀도 : ", precision_score(y_test, pred))
    print("재현율 : ", recall_score(y_test, pred))
    print("오차행렬 : ", confusion_matrix(y_test, pred))
    print("f1-score : ", f1_score(y_test, pred))
    print("전체 결과 : ", classification_report(y_test, pred))

In [ ]:
metrics_eval(y_test, y_pred)

## 4.3.정밀도 재현율 트레이드오프
- 정밀도(Precision)와 재현율(Recall)은 서로 상충(trade-off)하는 관계를 가진다.
- 즉, 정밀도를 높이면 재현율이 낮아지고, 재현율을 높이면 정밀도가 낮아지는 경향이 있다.
- 예) 모델이 "긍정(Positive)"이라고 예측할 기준을 엄격하게 설정하면, 잘못된 긍정(FP)은 줄어들지만, 긍정을 아예 놓쳐버릴(FN 증가) 가능성이 커짐
- 특정 문제에서는 정밀도를 우선할지, 재현율을 우선할지 결정해야 함
- 비즈니스와 도메인 요구사항에 따라 적절한 균형(F1-score)을 찾는 것이 중요

In [ ]:
pred_proba = model.predict_proba(X_test)    # 분류 모델(Classifier)이 각 클래스에 대한 예측 확률(probability)을 반환하는 함수
pred = model.predict(X_test)

In [ ]:
# 예측 확률 배열의 형태 확인 (샘플 개수, 클래스 개수)
pred_proba.shape

In [ ]:
# np.concatenate()를 사용하여 예측 확률과 최종 예측값을 나란히 배치하기 위해 (샘플 개수, 1) 형태로 변환
pred.reshape(-1, 1).shape

In [ ]:
# 한눈에 확인해보기
# 배열 구조 [클래스1 예측 확률, 클래스2 예측 확률, 예측 클래스]
pred_proba_result = np.concatenate([pred_proba, pred.reshape(-1, 1)], axis=1)
pred_proba_result[:10]

## 4.4.precision_recall_curve() 확인해보기

In [ ]:
from sklearn.metrics import precision_recall_curve

# 레이블이 1일때의 예측 확률을 추출
pred_proba_class1 = model.predict_proba(X_test)[:, 1]

In [ ]:
# Precision-Recall Curve 계산
# thresholds: Precision과 Recall을 계산한 다양한 임곗값 리스트
precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_class1)

In [ ]:
# 임곗값(Threshold)에 따라 Precision이 어떻게 변하는지 확인 가능
precisions

In [ ]:
# 차트로 그려보기
plt.figure(figsize=(8, 6))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="recalls")
plt.xlabel("Threshold")
plt.ylabel("precision&recall")
plt.legend()
plt.show()

- 비즈니스 목표에 따라 Precision과 Recall 중 무엇을 우선할지 정해야 한다.
    - 정밀도(Precision)를 우선하는 경우
    - 잘못된 긍정 예측(False Positive, FP)을 줄이는 것이 중요한 경우
        - 예: 사기 탐지, 암 진단, 위험한 상황 감지
    - 재현율(Recall)을 우선하는 경우
    - 실제 양성(True Positive)을 놓치는 것이 치명적인 경우
        - 예: 스팸 필터링, 질병 진단, 불량품 검사

 ## 4.5.Roc 곡선과 AUC 점수를 그래프로 나타내기
- ROC(Receiver Operating Characteristic) 곡선은 분류 모델의 성능을 평가하는 그래프이다.
- 이 곡선은 모델의 민감도(Sensitivity, TPR, True Positive Rate) 와 1-특이도(1-Specificity, FPR, False Positive Rate) 를 다양한 기준(Threshold) 값에 따라 변화시키면서 그리는 그래프이다.



 - FPR(False Positive Rate) : 실제 네거티브인 샘플 중에서 Positive 로 잘못 예측한 비율
 - TPR(True Positive Rate) : 실제 Positive 인 샘플중에서 제대로 예측한 비율 : 재현율
 - AUC : ROC 곡선의 아래면적을 뜻함.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ROC curve 를 위한 FPR, TPR 계산
fprs, tprs, roc_thresholds = roc_curve(y_test, y_pred)

In [ ]:
# AUC 점수를 계산
# AUC 값이 1에 가까울수록 모델 성능이 좋음 (최대 1, 랜덤이면 0.5)
roc_auc = roc_auc_score(y_test, pred_proba_class1)      # AUC(ROC 곡선 아래 면적)를 계산
print("roc_auc 점수는 :", roc_auc)

In [ ]:
## 차트 그려보기
plt.figure(figsize=(8, 6))
plt.plot(fprs, tprs, label=f"ROC Curve. AUC value: {roc_auc:.2f}")
plt.plot([0,1], [0, 1], label="Random")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.show()